# Debug Core Detection

This notebook helps debug and tune the core detection parameters.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
# Load image
image_path = 'path/to/your/core_image.jpg'
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(f"Image shape: {image.shape}")
plt.figure(figsize=(15, 10))
plt.imshow(image_rgb)
plt.title('Original Image')
plt.axis('off')
plt.show()

## Method 1: Simple Color-Based Segmentation

Core samples are typically lighter (beige/tan) compared to the dark background.

In [ ]:
# Convert to different color spaces
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

# Display different channels
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].imshow(image_rgb)
axes[0, 0].set_title('Original RGB')
axes[0, 0].axis('off')

axes[0, 1].imshow(gray, cmap='gray')
axes[0, 1].set_title('Grayscale')
axes[0, 1].axis('off')

axes[0, 2].imshow(hsv[:,:,0], cmap='hsv')
axes[0, 2].set_title('HSV - Hue')
axes[0, 2].axis('off')

axes[1, 0].imshow(hsv[:,:,1], cmap='gray')
axes[1, 0].set_title('HSV - Saturation')
axes[1, 0].axis('off')

axes[1, 1].imshow(hsv[:,:,2], cmap='gray')
axes[1, 1].set_title('HSV - Value')
axes[1, 1].axis('off')

axes[1, 2].imshow(lab[:,:,0], cmap='gray')
axes[1, 2].set_title('LAB - Lightness')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

# Print intensity statistics
print(f"Grayscale - Min: {gray.min()}, Max: {gray.max()}, Mean: {gray.mean():.2f}")
print(f"HSV Value - Min: {hsv[:,:,2].min()}, Max: {hsv[:,:,2].max()}, Mean: {hsv[:,:,2].mean():.2f}")

In [ ]:
# Try simple thresholding - cores are lighter than background
# Adjust this threshold based on your image
threshold_value = 100  # Adjust this!

_, binary = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Grayscale')
axes[0].axis('off')

axes[1].imshow(binary, cmap='gray')
axes[1].set_title(f'Binary Threshold = {threshold_value}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Clean up the binary image with morphological operations
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))

# Remove small noise
opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)

# Fill small holes
closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(binary, cmap='gray')
axes[0].set_title('Original Binary')
axes[0].axis('off')

axes[1].imshow(opening, cmap='gray')
axes[1].set_title('After Opening (noise removal)')
axes[1].axis('off')

axes[2].imshow(closing, cmap='gray')
axes[2].set_title('After Closing (hole filling)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Find contours on cleaned binary image
contours, hierarchy = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Found {len(contours)} contours")

# Analyze contour properties
contour_info = []
for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    x, y, w, h = cv2.boundingRect(contour)
    aspect_ratio = w / h if h > 0 else 0
    contour_info.append({
        'id': i,
        'area': area,
        'x': x, 'y': y, 'w': w, 'h': h,
        'aspect_ratio': aspect_ratio
    })

# Sort by area
contour_info.sort(key=lambda x: x['area'], reverse=True)

# Show top 20 largest contours
print("\nTop 20 largest contours:")
print(f"{'ID':<5} {'Area':<10} {'Width':<8} {'Height':<8} {'Aspect':<8}")
print("-" * 50)
for info in contour_info[:20]:
    print(f"{info['id']:<5} {info['area']:<10.0f} {info['w']:<8} {info['h']:<8} {info['aspect_ratio']:<8.2f}")

In [ ]:
# Visualize ALL contours
vis_image = image_rgb.copy()
cv2.drawContours(vis_image, contours, -1, (0, 255, 0), 2)

plt.figure(figsize=(15, 10))
plt.imshow(vis_image)
plt.title(f'All {len(contours)} Detected Contours')
plt.axis('off')
plt.show()

In [ ]:
# Filter contours based on properties
# Adjust these parameters!
MIN_AREA = 2000          # Minimum area in pixels
MAX_AREA = 500000        # Maximum area in pixels
MIN_ASPECT_RATIO = 1.2   # Cores are elongated horizontally
MAX_ASPECT_RATIO = 30    # But not too elongated
MIN_WIDTH = 50           # Minimum width
MIN_HEIGHT = 20          # Minimum height

valid_contours = []
valid_boxes = []

for info in contour_info:
    area = info['area']
    w = info['w']
    h = info['h']
    aspect_ratio = info['aspect_ratio']
    
    # Apply filters
    if (MIN_AREA <= area <= MAX_AREA and
        MIN_ASPECT_RATIO <= aspect_ratio <= MAX_ASPECT_RATIO and
        w >= MIN_WIDTH and h >= MIN_HEIGHT):
        
        valid_contours.append(contours[info['id']])
        valid_boxes.append((info['x'], info['y'], info['w'], info['h']))

print(f"\nFiltered to {len(valid_contours)} valid core pieces")
print(f"\nFilter criteria:")
print(f"  Area: {MIN_AREA} - {MAX_AREA} pixels")
print(f"  Aspect ratio: {MIN_ASPECT_RATIO} - {MAX_ASPECT_RATIO}")
print(f"  Min width: {MIN_WIDTH}px, Min height: {MIN_HEIGHT}px")

In [ ]:
# Visualize filtered contours
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(image_rgb)

for i, (x, y, w, h) in enumerate(valid_boxes):
    rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x + 5, y + 15, str(i+1), color='yellow', fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.set_title(f'Detected {len(valid_boxes)} Core Pieces')
ax.axis('off')
plt.tight_layout()
plt.show()

## Method 2: Edge-Based Detection

Detect cores based on edges and boundaries.

In [ ]:
# Edge detection
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 30, 100)

# Dilate edges to connect nearby edges
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
dilated_edges = cv2.dilate(edges, kernel, iterations=2)

# Close gaps
closed_edges = cv2.morphologyEx(dilated_edges, cv2.MORPH_CLOSE, kernel, iterations=3)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Original Grayscale')
axes[0, 0].axis('off')

axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title('Canny Edges')
axes[0, 1].axis('off')

axes[1, 0].imshow(dilated_edges, cmap='gray')
axes[1, 0].set_title('Dilated Edges')
axes[1, 0].axis('off')

axes[1, 1].imshow(closed_edges, cmap='gray')
axes[1, 1].set_title('Closed Edges')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Method 3: Row-by-Row Processing

Process each horizontal row separately.

In [ ]:
# Detect rows using horizontal projection
h_projection = np.mean(gray, axis=1)

# Plot horizontal projection
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

ax1.plot(h_projection, range(len(h_projection)))
ax1.set_ylim(len(h_projection), 0)
ax1.set_xlabel('Average Intensity')
ax1.set_ylabel('Y Position (pixels)')
ax1.set_title('Horizontal Projection')
ax1.grid(True)

ax2.imshow(image_rgb)
ax2.set_title('Original Image')
ax2.axis('off')

plt.tight_layout()
plt.show()

# Find peaks and valleys in projection
from scipy.signal import find_peaks

# Find valleys (dark separator lines)
valleys, _ = find_peaks(-h_projection, height=-150, distance=30)

print(f"Found {len(valleys)} potential row separators at y-positions:")
print(valleys)

In [ ]:
# Visualize detected rows
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(image_rgb)

for y in valleys:
    ax.axhline(y=y, color='red', linewidth=2, linestyle='--')

ax.set_title(f'Detected {len(valleys)} Row Separators')
ax.axis('off')
plt.tight_layout()
plt.show()

## Interactive Parameter Tuning

Use this cell to quickly test different threshold values.

In [ ]:
def test_detection(threshold=100, min_area=2000, max_area=500000, 
                   min_aspect=1.2, max_aspect=30, kernel_size=7, iterations=2):
    """
    Test detection with different parameters.
    """
    # Threshold
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    
    # Morphological operations
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=iterations)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=iterations+1)
    
    # Find contours
    contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Filter contours
    valid_boxes = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        aspect_ratio = w / h if h > 0 else 0
        
        if (min_area <= area <= max_area and
            min_aspect <= aspect_ratio <= max_aspect):
            valid_boxes.append((x, y, w, h))
    
    # Sort by position (top to bottom, left to right)
    valid_boxes.sort(key=lambda box: (box[1], box[0]))
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    axes[0].imshow(closing, cmap='gray')
    axes[0].set_title('Binary Mask')
    axes[0].axis('off')
    
    axes[1].imshow(image_rgb)
    for i, (x, y, w, h) in enumerate(valid_boxes):
        rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x + 5, y + 15, str(i+1), color='yellow', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    axes[1].set_title(f'Detected {len(valid_boxes)} Pieces')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Parameters: threshold={threshold}, min_area={min_area}, aspect={min_aspect}-{max_aspect}")
    print(f"Detected {len(valid_boxes)} core pieces")
    
    return valid_boxes

# Test with different parameters - ADJUST THESE!
boxes = test_detection(
    threshold=100,      # Adjust based on your image brightness
    min_area=2000,      # Minimum piece size
    max_area=500000,    # Maximum piece size
    min_aspect=1.2,     # Minimum width/height ratio
    max_aspect=30,      # Maximum width/height ratio
    kernel_size=7,      # Morphology kernel size
    iterations=2        # Morphology iterations
)

## Tips for Tuning:

1. **If detecting too many small pieces**: Increase `min_area`
2. **If missing pieces**: Lower `threshold` or `min_area`
3. **If detecting background**: Increase `min_aspect` or adjust `threshold`
4. **If pieces are fragmented**: Increase `iterations` or `kernel_size`
5. **If pieces are merged**: Decrease `iterations`

Try different values in the cell above until you get good results!